# Phase 3a Experiments: Local Alignment Activation

このノートブックでは、Phase 3aで実装したローカルアライメント活性化の効果を検証します。

## 実験設定

### 設定A: パラメータ調整版
- global_threshold: 0.75 (0.6から引き上げ)
- local_min_identity: 0.45 (0.5から引き下げ)
- min_local_length: 10 (5から引き上げ)
- large_seq_threshold: 500
- force_local_for_large: True
- min_coverage_for_skip_local: 0.7

### 設定B: 現状設定（ベースライン）
- global_threshold: 0.6
- その他デフォルト値

## Settings

In [1]:
import sys
import os
from pathlib import Path
from tqdm.notebook import tqdm

In [2]:
# ===== Environment detection =====
IS_KAGGLE = "KAGGLE_URL_BASE" in os.environ

print("Running on Kaggle:", IS_KAGGLE)


# ===== Path settings =====
if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/stanford-rna-3d-folding-2")
    SRC_DIR = Path("/kaggle/input/rna2-source-codes")
    WORK_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path.cwd().parent
    DATA_DIR = BASE_DIR / "data" / "stanford-rna-3d-folding-2"
    SRC_DIR = BASE_DIR / "src"
    SCRIPT_DIR = BASE_DIR / "scripts"
    WORK_DIR = BASE_DIR / "data" / "output"
    EXP_DIR = BASE_DIR / "experiments"


print("DATA_DIR:", DATA_DIR)
print("SRC_DIR:", SRC_DIR)
print("WORK_DIR:", WORK_DIR)

Running on Kaggle: False
DATA_DIR: /Users/tatsuki/work/kaggle/kauto/competitions/rna2/data/stanford-rna-3d-folding-2
SRC_DIR: /Users/tatsuki/work/kaggle/kauto/competitions/rna2/src
WORK_DIR: /Users/tatsuki/work/kaggle/kauto/competitions/rna2/data/output


In [3]:
# ===== File paths =====
train_seq_path = DATA_DIR / "train_sequences.csv"
train_label_path = DATA_DIR / "train_labels.csv"
valid_seq_path = DATA_DIR / "validation_sequences.csv"
valid_label_path = DATA_DIR / "validation_labels.csv"

print("Train seq:", train_seq_path.exists())
print("Train label:", train_label_path.exists())
print("Val seq:", valid_seq_path.exists())

Train seq: True
Train label: True
Val seq: True


In [4]:
# ===== Imports =====
sys.path.append(str(SRC_DIR))
from baseline.data import load_sequences, load_labels
from baseline.template_model import TemplateRepository
from baseline.search import seq_identity
from baseline.predict import generate_submission

if not IS_KAGGLE:
    sys.path.append(str(SCRIPT_DIR))
    from evaluate import evaluate_submission

## Load Data

In [5]:
# ===== Load data =====
train_seq = load_sequences(train_seq_path)
train_labels = load_labels(train_label_path)

valid_seq = load_sequences(valid_seq_path)
valid_labels = load_labels(valid_label_path)

print("Train size:", len(train_seq))
print("Val size:", len(valid_seq))

Train size: 5716
Val size: 28


## Create template repository

In [6]:
# Fit repository and generate a submission
repo = TemplateRepository()

repo.fit(train_seq, train_labels)
print('templates count:', len(repo.templates))

templates count: 5716


## 実験A: パラメータ調整版

In [7]:
# Configuration A: Tuned parameters
cfg_a = {
    'fill_gaps': True,
    'max_interp_length': 10,
    'interp_method': 'linear',
    'neighbor_k': 2,
    'use_kabsch': True,
    # Phase 3a parameters
    'global_threshold': 0.75,
    'local_min_identity': 0.45,
    'min_local_length': 10,
    'large_seq_threshold': 500,
    'force_local_for_large': True,
    'min_coverage_for_skip_local': 0.7,
}
repo.config = cfg_a
print('Config A:', repo.config)

Config A: {'fill_gaps': True, 'max_interp_length': 10, 'interp_method': 'linear', 'neighbor_k': 2, 'use_kabsch': True, 'global_threshold': 0.75, 'local_min_identity': 0.45, 'min_local_length': 10, 'large_seq_threshold': 500, 'force_local_for_large': True, 'min_coverage_for_skip_local': 0.7}


In [8]:
# Run prediction with configuration A
if not IS_KAGGLE:
    output_dir_a = EXP_DIR / "diagnostics" / "phase3a_config_a"
    output_dir_a.mkdir(parents=True, exist_ok=True)
    
    valid_pred_a = generate_submission(
        valid_seq, repo, seq_identity, n_structures=5, n_jobs=4,
        diagnostic_output_path=output_dir_a / "prediction_log.jsonl"
    )
    print('Config A - submission rows:', len(valid_pred_a))
else:
    valid_pred_a = generate_submission(
        valid_seq, repo, seq_identity, n_structures=5, n_jobs=4
    )
    print('Config A - submission rows:', len(valid_pred_a))

Config A - submission rows: 9762


In [9]:
# Evaluate configuration A
if not IS_KAGGLE:
    df_a, summary_a = evaluate_submission(pred_df=valid_pred_a, true_df=valid_labels)
    
    print('='*60)
    print('Config A Validation Summary:')
    print('='*60)
    print(f"Mean RMSD: {summary_a['mean_rmsd']:.2f} Å")
    print(f"Median RMSD: {summary_a['median_rmsd']:.2f} Å")
    print(f"Evaluated targets: {summary_a['n_evaluated']}/{summary_a['n_targets']}")
    
    # Save results
    df_a.to_csv(output_dir_a / 'validation_rmsd_results.csv', index=False)
    print(f"\nResults saved to {output_dir_a}")

Config A Validation Summary:
Mean RMSD: 35.69 Å
Median RMSD: 29.72 Å
Evaluated targets: 28/28

Results saved to /Users/tatsuki/work/kaggle/kauto/competitions/rna2/experiments/diagnostics/phase3a_config_a


## 実験B: 現状設定（ベースライン）

In [10]:
# Configuration B: Current baseline
cfg_b = {
    'fill_gaps': True,
    'max_interp_length': 10,
    'interp_method': 'linear',
    'neighbor_k': 2,
    'use_kabsch': True,
    # Phase 3a parameters (default values)
    'global_threshold': 0.6,
    'local_min_identity': 0.5,
    'min_local_length': 5,
    'large_seq_threshold': 500,
    'force_local_for_large': True,
    'min_coverage_for_skip_local': 0.7,
}
repo.config = cfg_b
print('Config B:', repo.config)

Config B: {'fill_gaps': True, 'max_interp_length': 10, 'interp_method': 'linear', 'neighbor_k': 2, 'use_kabsch': True, 'global_threshold': 0.6, 'local_min_identity': 0.5, 'min_local_length': 5, 'large_seq_threshold': 500, 'force_local_for_large': True, 'min_coverage_for_skip_local': 0.7}


In [11]:
# Run prediction with configuration B
if not IS_KAGGLE:
    output_dir_b = EXP_DIR / "diagnostics" / "phase3a_config_b"
    output_dir_b.mkdir(parents=True, exist_ok=True)
    
    valid_pred_b = generate_submission(
        valid_seq, repo, seq_identity, n_structures=5, n_jobs=4,
        diagnostic_output_path=output_dir_b / "prediction_log.jsonl"
    )
    print('Config B - submission rows:', len(valid_pred_b))
else:
    valid_pred_b = generate_submission(
        valid_seq, repo, seq_identity, n_structures=5, n_jobs=4
    )
    print('Config B - submission rows:', len(valid_pred_b))

Config B - submission rows: 9762


In [12]:
# Evaluate configuration B
if not IS_KAGGLE:
    df_b, summary_b = evaluate_submission(pred_df=valid_pred_b, true_df=valid_labels)
    
    print('='*60)
    print('Config B Validation Summary:')
    print('='*60)
    print(f"Mean RMSD: {summary_b['mean_rmsd']:.2f} Å")
    print(f"Median RMSD: {summary_b['median_rmsd']:.2f} Å")
    print(f"Evaluated targets: {summary_b['n_evaluated']}/{summary_b['n_targets']}")
    
    # Save results
    df_b.to_csv(output_dir_b / 'validation_rmsd_results.csv', index=False)
    print(f"\nResults saved to {output_dir_b}")

Config B Validation Summary:
Mean RMSD: 35.94 Å
Median RMSD: 29.89 Å
Evaluated targets: 28/28

Results saved to /Users/tatsuki/work/kaggle/kauto/competitions/rna2/experiments/diagnostics/phase3a_config_b


## 比較分析

In [13]:
# Compare results
if not IS_KAGGLE:
    import pandas as pd
    
    print('='*60)
    print('COMPARISON: Config A vs Config B')
    print('='*60)
    
    comparison = pd.DataFrame([
        {
            'Config': 'A (Tuned)',
            'Mean RMSD': f"{summary_a['mean_rmsd']:.2f}",
            'Median RMSD': f"{summary_a['median_rmsd']:.2f}",
            'Improvement': '-'
        },
        {
            'Config': 'B (Baseline)',
            'Mean RMSD': f"{summary_b['mean_rmsd']:.2f}",
            'Median RMSD': f"{summary_b['median_rmsd']:.2f}",
            'Improvement': '-'
        },
        {
            'Config': 'Delta (A - B)',
            'Mean RMSD': f"{summary_a['mean_rmsd'] - summary_b['mean_rmsd']:.2f}",
            'Median RMSD': f"{summary_a['median_rmsd'] - summary_b['median_rmsd']:.2f}",
            'Improvement': f"{((summary_b['mean_rmsd'] - summary_a['mean_rmsd']) / summary_b['mean_rmsd'] * 100):.1f}%"
        }
    ])
    
    print(comparison.to_string(index=False))
    print()
    print('Note: Negative delta = improvement (lower RMSD is better)')

COMPARISON: Config A vs Config B
       Config Mean RMSD Median RMSD Improvement
    A (Tuned)     35.69       29.72           -
 B (Baseline)     35.94       29.89           -
Delta (A - B)     -0.24       -0.17        0.7%

Note: Negative delta = improvement (lower RMSD is better)
